## Библиотеки и константы


In [1]:
import pandas as pd
import numpy as np
import math
import matplotlib
import matplotlib.pyplot as plt
import random
import scipy.stats as stats
import pylab
import struct
import requests
import json
import os
from IPython.display import clear_output
from math import factorial
from collections import defaultdict
from itertools import product, combinations, permutations
import inspect

%matplotlib inline

In [2]:
n_exp = 10000

n_exp = 500
n_discs = 5000
n_substats = 4

TOTAL_COUNT = 1_000_000
BATCH_SIZE = 10_000

ARTIFACT_FORMAT = "<10i"
ARTIFACT_SIZE = struct.calcsize(ARTIFACT_FORMAT)

### Базовые значения

In [3]:
stats_map = {
    "HP%": 0,
    "HP": 1, 
    "ATK%": 2,
    "ATK": 3,
    "DEF%": 4,
    "DEF": 5,
    "PEN": 6,
    "AP": 7, # Anomaly Proficiency
    "CRIT_RATE": 8,
    "CRIT_DMG": 9,
    "PEN%": 10,
    "AM": 11,
    "Impact": 12,
    "ER": 13,
    "Physical Bonus": 14,
    "Fire Bonus": 15,
    "Ice Bonus": 16,
    "Electric Bonus": 17,
    "Ether Bonus": 18,
    "Wind Bonus": 19,
    "None": 100,
}

stat_names = [""] * (max(stats_map.values()) + 1)
for name, stat_id in stats_map.items():
    if stat_id >= 0:
        stat_names[stat_id] = name

main_stat_values_map = {
    stats_map["HP%"]: 3000,
    stats_map["HP"]: 2200, 
    stats_map["ATK%"]: 3000,
    stats_map["ATK"]: 316,
    stats_map["DEF%"]: 4800,
    stats_map["DEF"]: 184, 
    stats_map["AP"]: 92,
    stats_map["CRIT_RATE"]: 2400,
    stats_map["CRIT_DMG"]: 4800,
    stats_map["PEN%"]: 2400,
    stats_map["Physical Bonus"]: 3000, # same for all elements
    stats_map["AM"]: 3000,
    stats_map["ER"]: 6000,
    stats_map["Impact"]: 1800
}

base_sub_stats_choices = np.array([
    stats_map["HP%"],
    stats_map["HP"], 
    stats_map["ATK%"],
    stats_map["ATK"],
    stats_map["DEF%"],
    stats_map["DEF"],
    stats_map["PEN"],
    stats_map["AP"],
    stats_map["CRIT_RATE"],
    stats_map["CRIT_DMG"]
], dtype=np.int32)

# 100%, 90%, 80%, 70%
sub_stat_values = np.array([
    300, # HP%
    112, # HP
    300, # ATK%
    19, # ATK
    480, # DEF%
    15, # DEF
    9, # PEN
    9, # AP
    240, # CRIT_RATE
    480, # CRIT_DMG
              
], dtype=np.int32)

sub_stats_weights = np.array([
    113, # HP%
    103, # HP
    90, # ATK%
    98, # ATK
    131, # DEF%
    112, # DEF
    107, # PEN
    97, # AP
    116, # CRIT_RATE
    104, # CRIT_DMG
              
], dtype=np.int32)

enka_stat_map = {
    11102: stats_map["HP%"],
    11103: stats_map["HP"], 
    12102: stats_map["ATK%"],
    12103: stats_map["ATK"],
    13102: stats_map["DEF%"],
    13103: stats_map["DEF"], 
    31203: stats_map["AP"],
    20103: stats_map["CRIT_RATE"],
    21103: stats_map["CRIT_DMG"],
    23103: stats_map["PEN%"],
    23203: stats_map["PEN"],
    31503: stats_map["Physical Bonus"],
    31603: stats_map["Fire Bonus"],
    31703: stats_map["Ice Bonus"],
    31803: stats_map["Electric Bonus"],
    31903: stats_map["Ether Bonus"],
    32303: stats_map["Wind Bonus"],
    31402: stats_map["AM"],
    30502: stats_map["ER"],
    12202: stats_map["Impact"]
}

BASE_STAT_IDS = {
    11101: "HP",
    12101: "ATK",
    13101: "DEF"
}

file_names = {'HP%', 'HP', 'ATK%', 'ATK', 'DEF%', 'DEF', 'AP', "CRIT_RATE", "CRIT_DMG"}

profit_names = {name: 0.0 for name in stat_names[:10]}

In [4]:
slot_1_pop = [stats_map["HP"]]
slot_1_prob = np.array([1], dtype=float)

slot_2_pop = [stats_map["ATK"]]
slot_2_prob = np.array([1], dtype=float)

slot_3_pop = [stats_map["DEF"]]
slot_3_prob = np.array([1], dtype=float)

slot_4_pop = [
    stats_map["HP%"],
    stats_map["ATK%"],
    stats_map["DEF%"],
    stats_map["CRIT_RATE"],
    stats_map["CRIT_DMG"],
    stats_map["AP"]
]
slot_4_prob = np.array([1, 1, 1, 1, 1, 1], dtype=float)
slot_4_prob /= slot_4_prob.sum()

slot_5_pop = [
    stats_map["HP%"],
    stats_map["ATK%"],
    stats_map["DEF%"],
    stats_map["PEN%"],
    stats_map["Physical Bonus"],
    stats_map["Fire Bonus"],
    stats_map["Ice Bonus"],
    stats_map["Electric Bonus"],
    stats_map["Ether Bonus"],
    stats_map["Wind Bonus"]
]
slot_5_prob = np.array([2, 2, 2, 1, 1, 1, 1, 1, 1], dtype=float)
slot_5_prob /= slot_5_prob.sum()


slot_6_pop = [
    stats_map["HP%"],
    stats_map["ATK%"],
    stats_map["DEF%"],
    stats_map["AM"],
    stats_map["ER"],
    stats_map["Impact"]
]
slot_6_prob = np.array([2, 2, 2, 2, 2, 1], dtype=float)
slot_6_prob /= slot_6_prob.sum()

slots = ["FLOWER", "PLUME", "SANDS", "GOBLET", "CIRCLET"]

In [5]:
with open("utils/avatars.json", encoding="utf-8") as file:
    avatars_data = json.load(file)

with open("utils/names.json", encoding="utf-8") as file:
    names = json.load(file)
    
with open("utils/avatars.json", encoding="utf-8") as file:
    avatars_data = json.load(file)

with open("utils/weapons.json", encoding="utf-8") as file:
    weapons_data = json.load(file)
    
WEAPON_MAX_LEVEL_MULTIPLIER = (
    1
    + 94090 / 10000
    + 44610 / 10000
)

### Функции

In [11]:
def parse_character(avatar):
    name_ru = get_character_name(
        avatar,
        "ru"
    )

    base_stats = get_character_base_stats(avatar)

    discs = []

    for equipment in avatar["EquippedList"]:
        discs.append(
            parse_disc(equipment)
        )

    discs.sort(key=lambda disc: disc["slot"])

    return {
        "name_ru": name_ru,
        "base_stats": base_stats,
        "profit": [0.0] * 10,
        "score": 0.0,
        "discs": discs
    }

def parse_disc(slot):
    equipment = slot["Equipment"]
    main_property = equipment["MainPropertyList"][0]

    enka_main_id = main_property["PropertyId"]
    main_stat_id = enka_stat_map[enka_main_id]
    main_stat_value = main_property["PropertyValue"] * 4

    substats = [0] * 10

    for property_data in equipment["RandomPropertyList"]:
        enka_stat_id = property_data["PropertyId"]
        stat_id = enka_stat_map[enka_stat_id]

        # Здесь нам нужно именно число проков.
        # Точное поле подставим по реальному Enka JSON.
        roll_count = property_data["PropertyLevel"]

        substats[stat_id] = roll_count

    return {
        "slot": slot["Slot"],

        "main_stat": {
            "id": main_stat_id,
            "name": stat_names[main_stat_id],
            "value": main_stat_value
        },
        "substats": substats,
        "score": 0.0
    }

def get_character_name(avatar, language):
    avatar_id = str(avatar["Id"])
    avatar_data = avatars_data[avatar_id]

    name_key = avatar_data["Name"]

    return names[language][name_key]

def get_character_base_properties(avatar):
    avatar_data = avatars_data[str(avatar["Id"])]

    promotion_props = avatar_data["PromotionProps"][-1]
    enhancement_props = avatar_data["CoreEnhancementProps"][-1]

    result = {}

    for stat_id in BASE_STAT_IDS:
        stat_key = str(stat_id)

        base_value = avatar_data["BaseProps"][stat_key]
        growth_value = math.floor(avatar_data["GrowthProps"][stat_key] * 59 / 10000)
        promotion_value = promotion_props[stat_key]
        enhancement_value = enhancement_props.get(stat_key, 0)

        result[stat_id] = (
            base_value
            + growth_value
            + promotion_value
            + enhancement_value
        )

    return result

def get_weapon_base_properties(avatar):
    weapon = avatar.get("Weapon")

    if weapon is None:
        return {}

    weapon_id = str(weapon["Id"])
    weapon_data = weapons_data[weapon_id]

    main_stat = weapon_data["MainStat"]

    stat_id = main_stat["PropertyId"]
    value = math.floor(main_stat["PropertyValue"] * WEAPON_MAX_LEVEL_MULTIPLIER)

    return {
        stat_id: value
    }


def get_character_base_stats(avatar):
    base_stats = {
        "HP": 0,
        "ATK": 0,
        "DEF": 0
    }

    # базовые статы самого персонажа
    character_stats = get_character_base_properties(avatar)
    for enka_stat_id, value in character_stats.items():
        if enka_stat_id in BASE_STAT_IDS:
            base_stats[BASE_STAT_IDS[enka_stat_id]] += value

    # базовые статы W-Engine
    weapon_stats = get_weapon_base_properties(avatar)
    for enka_stat_id, value in weapon_stats.items():
        if enka_stat_id in BASE_STAT_IDS:
            base_stats[BASE_STAT_IDS[enka_stat_id]] += value
            
    return base_stats

In [9]:
def load_zzz_player(uid):
    url = f"https://enka.network/api/zzz/uid/{uid}"

    response = requests.get(
        url,
        headers={
            "User-Agent": "ArtifactAnalyzer/1.0"
        }
    )

    response.raise_for_status()

    return response.json()

def load_zzz_player_from_file(uid):
    with open(f"utils/{uid}.json", "r", encoding="utf-8") as file:
        data = json.load(file)
    return data


def parse_zzz_player(player, update_characters=None):
    if update_characters is None:
        update_characters = []

    uid = str(player["uid"])

    file_path = f"characters/{uid}.json"

    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as file:
            characters = json.load(file)
    else:
        characters = {}

    avatars = player["PlayerInfo"]["ShowcaseDetail"]["AvatarList"]

    for avatar in avatars:
        name_en = get_character_name(avatar, "en")

        parsed_character = parse_character(avatar)
        old_character = characters.get(name_en)

        if old_character is not None:
            # Уже есть и обновление не запрошено
            if old_character["score"] > 1:
                continue
            
            # сохраняем наши расчётные/ручные данные
            parsed_character["preferred_main_stats"] = old_character["preferred_main_stats"]
            parsed_character["profit"] = old_character["profit"]
            parsed_character["profit_names"] = old_character["profit_names"]
            parsed_character["score"] = old_character["score"]

        characters[name_en] = parsed_character

    os.makedirs("characters", exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(
            characters,
            file,
            ensure_ascii=False,
            indent=4
        )
        
def update_flat_substat_profit(uid):
    with open("utils/character_profit.json", "r", encoding="utf-8") as file:
        profits = json.load(file) 
    file_path = f"characters/{uid}.json"
    with open(file_path, "r", encoding="utf-8") as file:
        characters = json.load(file)
        
    for character_name, character in characters.items():
        base_stats = character["base_stats"]
        char_profit_names = profits[character_name]
        profit = character["profit"]
        
        profit = [
            char_profit_names[stat_name]
            for stat_name in stat_names
        ]

        profit[1] = round(profit[0] * sub_stat_values[1] / (base_stats["HP"] * sub_stat_values[0] / 10000), 2)
        profit[3] = round(profit[2] * sub_stat_values[3] / (base_stats["ATK"] * sub_stat_values[2] / 10000), 2)
        profit[5] = round(profit[4] * sub_stat_values[5] / (base_stats["DEF"] * sub_stat_values[4] / 10000), 2)
        
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(
            characters,
            file,
            ensure_ascii=False,
            indent=4
        )

In [119]:
player = load_zzz_player_from_file(1500018666)
parse_zzz_player(player)

In [10]:
player = load_zzz_player(1500018666)
parse_zzz_player(player)

In [ ]:
update_flat_substat_profit(1500018666)